# 2 · Single tool execution, then parallel

Demo 1 only *built* a request — it serialized tools but never actually ran one.
Here we do the real thing, in two steps that mirror how the loop thinks:

1. **A single tool call** — the model asks for one thing, we run it once.
2. **Parallel tool execution** — the model asks for several *independent*
   things in one message; we fan them out onto threads and collect the results.

We use the built-in **file, web and calculator** tools throughout. Below, a
single `calc` call is executed exactly once via `execute_one`.


In [2]:
:dep agent_loop = { path = "/home/christian/Sandbox/agent-loop" }
:dep serde_json = "1"

use agent_loop::tools::ToolRegistry;
use agent_loop::chat::ToolCall;
use agent_loop::executor::execute_one;

// The built-ins include file, web and calculator tools.
let registry = ToolRegistry::with_builtins();
println!("built-ins ({}): {}", registry.len(),
    registry.all().iter().map(|t| t.name.as_str()).collect::<Vec<_>>().join(", "));

// The model emitted exactly ONE tool call. Running a loop is, at bottom, just
// doing this: name + arguments -> registry lookup -> execute -> tool result.
let call = ToolCall {
    id: "call_calc_1".into(),
    name: "calc".into(),
    arguments: r#"{"expression": "(2 + 3) * 4 ^ 2 + sqrt(9)"}"#.into(),
};

let started = std::time::Instant::now();
let result = execute_one(&registry, &call);
let elapsed_ms = started.elapsed().as_secs_f64() * 1000.0;

println!("single call `{}` took {:.3} ms", call.name, elapsed_ms);
println!("result -> {}", result.content.as_deref().unwrap_or("<none>"));


built-ins (7): file_read, file_write, file_list, bash_run, web_fetch, web_search, calc
single call `calc` took 0.135 ms
result -> 83


### What "a single tool call" means

That one call is the **smallest unit of the entire loop**. Everything else is a
refinement of it:

```
model says  call(calc, {"expression":"(2+3)*4^2+sqrt(9)"})
                    │
                    ▼
              execute_one(registry, &call)
                    │  1. parse arguments
                    │  2. find tool by name
                    │  3. run its executor
                    ▼
            tool_result message -> fed back to the model
```

A **file** tool works identically — same shape, different target. Try a single
`file_list` call on the notebook directory:

> Note: the kernel's working directory is the `notebooks/` folder (JupyterLab
> was launched with `--notebook-dir notebooks`), so a relative `"notebooks"`
> path would resolve to `notebooks/notebooks` and fail. We pass the absolute
> path instead so the demo works no matter where the kernel starts.


In [3]:

// A single FILE call: same execute_one, different tool. We use the absolute
// notebooks path because the kernel CWD is the notebooks dir itself.
let file_call = ToolCall {
    id: "call_list_1".into(),
    name: "file_list".into(),
    arguments: r#"{"path": "/home/christian/Sandbox/agent-loop/notebooks"}"#.into(),
};
let out = execute_one(&registry, &file_call);
println!("file_list ->\n{}", out.content.as_deref().unwrap_or("<none>"));


file_list ->
.ipynb_checkpoints
00_welcome.ipynb
01_request_response_tools.ipynb
02_parallel_tool_execution.ipynb
03_dynamic_tools.ipynb
04_hooks.ipynb
05_hook_execution.ipynb
agent-loop-diagram-dark.svg


### Several independent calls arrive at once

A real model rarely returns exactly one `tool_calls` entry — it often returns
several *independent* ones in a single assistant message. They could be N file
reads, N web fetches, or N separate calculations. Because they don't depend on
each other, they are prime candidates for running **in parallel**.

The executor exposes two strategies:
- **sequential** — run one at a time, in order;
- **parallel** — fan out onto worker threads and collect results back in the
  original order.

This demo is **fully local** (no network): we register `N` independent compute
tools that each sleep `300 ms`, then run the same batch both ways and compare
the wall-clock time. The ordering of results is preserved in *both* modes —
parallelism buys speed, never reordering.


In [4]:


use agent_loop::tools::{Tool, ToolResult, s_required};
use agent_loop::executor::execute_many;
use serde_json::{json, Value};
use std::time::Duration;

// Register 4 independent tools. Each claims 300 ms of work.
let mut registry = ToolRegistry::new();
for i in 0..4usize {
    registry.register(Tool::new(
        format!("compute_{i}"),
        format!("Simulate compute task {i}"),
        s_required(json!({}), &[]),
        move |_| {
            std::thread::sleep(Duration::from_millis(300));
            Ok(ToolResult::ok(format!("task {i} done")))
        },
    ));
}

// The model "emitted" 4 tool calls in a single assistant message.
let calls: Vec<agent_loop::chat::ToolCall> = (0..4).map(|i| agent_loop::chat::ToolCall {
    id: format!("call_{i}"),
    name: format!("compute_{i}"),
    arguments: "{}".into(),
}).collect();
println!("model emitted {} tool calls in ONE message", calls.len());


model emitted 4 tool calls in ONE message


### Sequential run

Every call waits for the previous one: 4 × 300 ms ≈ **1200 ms**.


In [5]:

let (seq_results, seq_stats) = execute_many(&registry, &calls, false);
println!("sequential: {} calls took {:.1} ms", seq_stats.n_calls, seq_stats.elapsed_ms);
for r in &seq_results {
    println!("   ⇠ {}", r.content.as_deref().unwrap_or(""));
}


sequential: 4 calls took 1200.5 ms
   ⇠ task 0 done
   ⇠ task 1 done
   ⇠ task 2 done
   ⇠ task 3 done


()

### Parallel run

Workers run concurrently: ≈ **300 ms** for the whole batch, ~4× faster.
Results are returned in the *same order* as the calls.


In [6]:

let (par_results, par_stats) = execute_many(&registry, &calls, true);
println!("parallel  : {} calls took {:.1} ms", par_stats.n_calls, par_stats.elapsed_ms);
for r in &par_results {
    println!("   ⇠ {}", r.content.as_deref().unwrap_or(""));
}


parallel  : 4 calls took 301.1 ms
   ⇠ task 0 done
   ⇠ task 1 done
   ⇠ task 2 done
   ⇠ task 3 done


()

### Side-by-side

```
sequential : 1200 ms   ████████████████████████████
parallel   :  300 ms   ██████
```

The speedup scales with the number of *independent* calls. The loop chooses the
strategy via `AgentConfig.parallel_tools`. The same `execute_many` is what the
agent loop in demos 4–5 calls under the hood — it is just `execute_one` (from
the single-call step above) repeated, optionally on worker threads.
